# Logistic Regression + Sentence-BERT 문체 융합 (Kaggle)

## 1. 저장소·환경

In [ ]:
%cd /kaggle/working
!rm -rf LLM-name-tagging
!git clone -b haram https://github.com/seoultech-Machine-Learning-26-2/LLM-name-tagging.git
%cd /kaggle/working/LLM-name-tagging
!git log -1 --oneline

import os
assert os.path.exists('experiments/D_logistic_regression.py'), \
    'experiments/D_logistic_regression.py 가 없다. haram 브랜치가 push 됐는지 확인'
print('실험 코드 확인됨')

import platform, sys, numpy, pandas, sklearn, scipy
print(platform.platform(), '| Python', sys.version.split()[0])
print('numpy', numpy.__version__, '| pandas', pandas.__version__,
      '| sklearn', sklearn.__version__, '| scipy', scipy.__version__)
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo 'GPU 없음 (고전 모델은 CPU로 충분)'

## 2. Logistic Regression (D-1 ~ D-2)

In [ ]:
!python -u -W ignore -m experiments.D_logistic_regression

## 3. 결과

In [ ]:
import pandas as pd
pd.set_option('display.width', 220)

res = pd.read_csv('results/course/D/logistic_regression/val_results.csv')
cols = ['run_id','feature_set','train_model_BA','model_BA','accuracy','macro_f1',
        'log_loss','ece','sibling_acc','company_BA','converged','n_iter','fit_seconds']
display(res[[c for c in cols if c in res.columns]].round(4))

### 3-1. train/val 차이

In [ ]:
lr = res[res.model == 'LogisticRegression'].copy()
lr['gap'] = lr.train_model_BA - lr.model_BA
display(lr[['config','train_model_BA','model_BA','gap']].round(4)
          .sort_values('gap', ascending=False))

from IPython.display import Image, display as d
d(Image('results/course/D/logistic_regression/fig_config_comparison.png'))

### 3-2. 혼동행렬

In [ ]:
d(Image('results/course/D/logistic_regression/fig_confusion_val.png'))

### 3-3. 오분류 5개

In [ ]:
err = pd.read_csv('results/course/D/logistic_regression/val_errors_seed42.csv')
for _, r in err.iterrows():
    print(f"[{r.id}] 실제 {r.true_model}  ->  예측 {r.pred_model} "
          f"(확률 {r.pred_prob:.3f}, 정답에 준 확률 {r.true_prob:.3f}, "
          f"같은 개발사={r.same_company})")
    print('   ', r.text_head[:200], '\n')

### 3-4. 계수

In [ ]:
coef = pd.read_csv('results/course/D/logistic_regression/coefficients_style.csv')
print('계수 절댓값 상위에 자주 등장한 특징 (18개 모델 중 몇 번)')
display(coef.feature.value_counts().head(12).to_frame('등장 모델 수'))
print('\n모델별 1위 특징')
display(coef.sort_values('abs_coef', ascending=False)
            .groupby('model').head(1)[['model','feature','coef']]
            .sort_values('model').reset_index(drop=True))

## 4. Sentence-BERT 문체 융합 (D-3 ~ D-4)

In [ ]:
import torch
assert torch.cuda.is_available(), 'Settings에서 Accelerator를 GPU로 바꾸고 1절부터 다시 실행'
print(torch.__version__, torch.cuda.get_device_name(0))
!pip -q install -U sentence-transformers

In [ ]:
# D_fusion.py 는 다음 단계에서 작성한다. 작성 후 아래 주석을 풀고 실행.
# !python -u -W ignore -m experiments.D_fusion --encoder sentence-transformers/all-MiniLM-L6-v2

## 5. 결과 저장

In [ ]:
!zip -qr /kaggle/working/D_results.zip results/course/D preds/val_D_* preds/test_D_* 2>/dev/null
!ls -lh /kaggle/working/D_results.zip
!unzip -l /kaggle/working/D_results.zip | tail -25

## 6. test 1회 (팀 합의 후에만)

In [ ]:
# !python -u -W ignore -m experiments.D_logistic_regression --final combined_C10.0